# 🚀 LLM Gateway Explained — Build One With LiteLLM + LangChain

## 📚 What You'll Learn in This Repository

This repository covers:

- What is an LLM Gateway? — The problem it solves
- Why do we need it? — Real production pain points
- Core capabilities — Routing, fallbacks, caching, observability, cost tracking
- Practical implementation — Build one from scratch using LiteLLM
- Integration with LangChain — Plug the gateway into your agentic apps
- Production patterns — Logging, retries, multi-provider fallbacks

By the end, you'll have a working LLM gateway that routes between OpenAI, Anthropic, and Groq — with caching, fallbacks, and cost tracking built in. 🔥

## 🧠 Part 1: What is an LLM Gateway?

Think of an LLM Gateway as a smart middleware layer that sits between your application and multiple LLM providers (OpenAI, Anthropic, Google, Groq, Cohere, local models, etc.).

```
                    ┌─────────────────────────────┐
                    │       Your Application      │
                    │  (Chatbot, RAG, Agent, etc) │
                    └──────────────┬──────────────┘
                                   │
                                   ▼
                    ┌─────────────────────────────┐
                    │       LLM GATEWAY           │
                    │  • Routing                  │
                    │  • Fallbacks                │
                    │  • Caching                  │
                    │  • Rate Limiting            │
                    │  • Cost Tracking            │
                    │  • Observability            │
                    └──────┬─────┬─────┬─────┬────┘
                           │     │     │     │
                           ▼     ▼     ▼     ▼
                        OpenAI Claude Gemini Groq
```

### Without a Gateway (The Pain 😩)

- Different SDKs and APIs for every provider
- No fallback if one provider goes down
- No central place to track costs
- Hard to switch models without rewriting code
- No caching → paying twice for the same query

### With a Gateway (The Joy 😎)

- One unified API for 100+ providers
- Automatic fallbacks if a provider fails
- Centralized logging, cost tracking, rate limiting
- Swap models with a config change, no code rewrite
- Cache repeated queries → save money

## ⚙️ Part 2: Installation & Setup

We'll use:

- **LiteLLM** → the most popular open-source LLM gateway (supports 100+ providers)
- **LangChain** → for building agentic workflows on top of the gateway
- **python-dotenv** → for managing API keys

In [1]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [2]:
import litellm
litellm.suppress_debug_info = True

In [3]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")
print("Gemini key loaded:    ", "✅" if os.getenv("GEMINI_API_KEY") else "❌")

OpenAI key loaded:     ✅
Anthropic key loaded:  ❌
Groq key loaded:       ✅
Gemini key loaded:     ✅


# 🎯 Part 3: The Simplest LiteLLM Example — Unified API

The biggest pain point: every provider has a different SDK.

LiteLLM gives you one function — `completion()` — that works with all of them. Look at how clean this is:

In [ ]:
from litellm import completion

# Same code, different providers — just change the `model` string!

# call Ollama
response_ollama = completion(
    model="ollama/gemma3:1b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("🔵 Ollama:    ", response_ollama.choices[0].message.content)

# Call Groq (super fast inference)
response_groq = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("🟢 Groq:      ", response_groq.choices[0].message.content)

# Call Gemini (Google's LLM)
response_gemini = completion(
    model="gemini/gemini-2.5-flash",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)
print("🟡 Gemini:    ", response_gemini.choices[0].message.content)

🔵 Ollama:     Retrieval Augmented Generating is a technique where AI models retrieve information from external databases or knowledge sources to enhance their response generation, making them more knowledgeable and relevant.**
🟢 Groq:       In project management, RAG is an acronym for Red, Amber, and Green, a traffic light-like system used to categorize tasks or projects into three colors, which signify their status: Red for critical issues or delays, Amber for issues requiring attention, and Green for tasks meeting their objectives or milestones.
🟡 Gemini:     RAG enhances large language model outputs by retrieving relevant information from an external knowledge base to augment its input prompt before generating a response.


🎉 Notice: Same code, three different providers. This alone is huge — you can switch providers with a string change.

But a real LLM Gateway does much more.

In [8]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/llama-3.3-70b-versatile"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-2.5-flash"),
    ("🔴 Ollama",     "ollama/gemma3:1b")
]

# ONE loop. ONE function call. Multiple providers.
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : RAG (Retrieve, Augment, Generate) is a type of artificial intelligence model tha
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : RAG (Retrieval-Augmented Generation) enhances Large Language Models by first ret
🔴 Ollama       : Retrieval-Augmented Generation explains that Large Language Models (LLMs) are en


# 🛡️ Part 4: Automatic Fallbacks — When OpenAI Goes Down

Real story: OpenAI had a 4-hour outage in November 2023. Apps that hard-coded gpt-4 went completely dark.

With a gateway, if one provider fails, we automatically fall back to another. Production apps must have this.

In [9]:
from litellm import completion

# Define a fallback chain: 
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "ollama/gemma3:1b",
        "groq/llama-3.3-70b-versatile"
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

16:15:01 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2837, in async_completion
    response = await client.post(
               ^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 289, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\ll

Response: Okay, let's dive into what an LLM (Large Language Model) gateway does. It simplifies bringing and managing these powerful models – like GPT-3 or Llama-2– to various applications without requiring deep ...

Which model actually answered? ollama/gemma3:1b


In [10]:
from litellm import completion

# Force the primary to fail by using a fake model name
# Then watch the fallback chain rescue the call
response = completion(
    model="openai/fake-nonexistent-model-9999",     # 👈 will fail intentionally
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[                              # 1st backup: real OpenAI model
        "groq/llama-3.3-70b-versatile",
                    "ollama/gemma3:1b"             # 2nd backup: Ollama
    ]
)

print("✅ App still got a response, even though the primary failed!")
print(f"\n🤖 Model that actually answered: {response.model}")
print(f"\n📝 Response: {response.choices[0].message.content[:200]}...")

16:18:18 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model openai/fake-nonexistent-model-9999: litellm.NotFoundError: OpenAIException - The model `fake-nonexistent-model-9999` does not exist or you do not have access to it.
Traceback (most recent call last):
  File "c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 886, in acompletion
    headers, response = await self.make_openai_chat_completion_request(
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 289, in async_wrapper
    result = await func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\llms\openai\openai.py", line 447, in make_openai_chat_completion_request
    raise e
  File "c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packa

✅ App still got a response, even though the primary failed!

🤖 Model that actually answered: llama-3.3-70b-versatile

📝 Response: An LLM (Large Language Model) Gateway is a software intermediary that enables access to and interaction with large language models (LLMs) over the internet or within an organization's network. The gat...


# 💰 Part 5: Cost Tracking — Know Where Your Money Goes

LiteLLM `automatically calculates the cost` of every call using its built-in pricing database. No more surprise bills.

In [11]:
from litellm import completion, completion_cost

response = completion(
    model="gemini/gemini-2.5-flash",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Get the exact USD cost of this single call
cost = completion_cost(completion_response=response)

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")

Response:     Code learns, then thinks,
Digital mind starts to awaken,
Future takes its form.

Input tokens:  8
Output tokens: 762
Cost:         $0.00190740


# ⚡ Part 6: Caching — Don't Pay Twice for the Same Question

If 100 users ask "What is RAG?", you don't need to call the LLM 100 times.

Enable in-memory caching with one line:

In [12]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("✅ LiteLLM state reset — ready for clean caching demo")

✅ LiteLLM state reset — ready for clean caching demo


In [14]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "what is the use of LLM Gateways ? Answer in three line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"❄️  First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="groq/llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

❄️  First call (API):   0.57s — LLM (Large Language Model) gateways act as interfaces between LLMs and external applications or services. They enable secure and controlled access to LLMs, allowing users to interact with the models while maintaining data privacy and security. LLM gateways also provide features like authentication, authorization, and rate limiting to manage LLM usage and prevent abuse.
⚡ Second call (cache): 0.0010s — LLM (Large Language Model) gateways act as interfaces between LLMs and external applications or services. They enable secure and controlled access to LLMs, allowing users to interact with the models while maintaining data privacy and security. LLM gateways also provide features like authentication, authorization, and rate limiting to manage LLM usage and prevent abuse.

🚀 Speedup: 561.4x faster, and ZERO cost on the second call!


# 🔀 Part 7: Smart Routing — The Right Model for the Right Job

## Why use one model for everything?

- **Coding tasks** → Claude Sonnet
- **Cheap summaries** → GPT-4o-mini
- **Super fast replies** → Groq Llama
- **Complex reasoning** → Claude Opus

Use LiteLLM's **Router** to define routing rules.

In [16]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",                              
        "litellm_params": {
            "model": "gemini/gemini-2.5-flash",                                 
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "ollama/gemma3:4b",
        }
    }
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}]
)

print("⚡ Fast/cheap (Groq): ", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (GPT-4o):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq):  AI (Artificial Intelligence) is transforming the software development industry in several ways:

1. **Automated Coding**: AI-powered tools can generat

🧠 Smart/coding (GPT-4o):
 You can reverse a string in Python using several methods. Here are a few common and idiomatic ways, wrapped in a Python function.

---

### Method 1: Using Slicing (The Most Pythonic Way)

This is generally considered the most concise and Pythonic way to reverse a string.

```python
def reverse_stri


💡 **Key insight:** Your app calls "fast-cheap" or "smart-coding" — abstract names. The router decides which provider to actually use. Tomorrow, you can swap Groq for a cheaper provider with `zero code changes`.

# 🔁 Part 8: Load Balancing Across Multiple API Keys

Hit rate limits on one OpenAI key? Add more keys to the same alias — the router load-balances automatically.

In [17]:
from litellm import Router
import os

# Two deployments under the same alias
# A pool of "smart" models — all equally capable, just different providers
model_list = [
    {
        "model_name": "gemini-pool",
        "litellm_params": {
            "model": "gemini/gemini-2.5-flash",
            "api_key": os.getenv("GEMINI_API_KEY"),
        },
        "model_info": {"id": "gemini-2.5-flash"}
    },
    
    {
        "model_name": "groq-pool",
        "litellm_params": {
            "model": "groq/llama-3.3-70b-versatile",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-llama-70b"}
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gemini-pool",
        messages=[{"role": "user", "content": f"Say hello, request {i+1}"}]
    )
    # Pull out which deployment served this request
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9}{deployment_id:<22}{latency:>6.0f} ms   {answer}")

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------


Task was destroyed but it is pending!
task: <Task pending name='Task-172' coro=<AsyncHTTPHandler.close() running at c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py:568>>
Task was destroyed but it is pending!
task: <Task pending name='Task-173' coro=<AsyncHTTPHandler.close() running at c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py:568>>
Task was destroyed but it is pending!
task: <Task pending name='Task-174' coro=<AsyncHTTPHandler.close() running at c:\Users\Rahul\LLM_Gateway-Learning\.venv\Lib\site-packages\litellm\llms\custom_httpx\http_handler.py:568>>


#1        gemini-2.5-flash        1373 ms   Hello, request 1
#2        gemini-2.5-flash        1432 ms   Hello, request 2
#3        gemini-2.5-flash        1437 ms   Hello! 3
#4        gemini-2.5-flash        1031 ms   Hello!
Hello!
Hello!
Hello!
#5        gemini-2.5-flash        1147 ms   Hello, request 5
#6        gemini-2.5-flash        1547 ms   Hello! Here is 6.


#### 🎯 Strategy 1: least-busy —

The "Express Checkout" PatternThe idea: Like picking the shortest line at a supermarket. The router tracks how many requests are currently in flight to each deployment and sends the new request to whichever one is least busy.

In [18]:
import os
from litellm import Router
from collections import Counter

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-2.5-flash",
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🔵 Gemini"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"   
)

hits = Counter()
for i in range(8):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

print("\n🎯 Distribution:")
for k, v in hits.most_common():
    print(f"   {k}: {'█' * v} ({v})")

Request 1 → 🔵 Gemini
Request 2 → 🔵 Gemini
Request 3 → 🔵 Gemini
Request 4 → 🔵 Gemini
Request 5 → 🔵 Gemini
Request 6 → 🔵 Gemini
Request 7 → 🔵 Gemini
Request 8 → 🟢 Groq

🎯 Distribution:
   🔵 Gemini: ███████ (7)
   🟢 Groq: █ (1)


#### 🎯 Strategy 2: latency-based-routing —

The "Always Pick the Fastest" Pattern The idea: The router measures the response time of each deployment over recent calls and sends new requests to whichever has been fastest. Speed wins.

In [19]:
import os
from litellm import Router
import time

model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-2.5-flash",
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🔵 Gemini"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama-3.3"}},
    
]

router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"   # 👈 picks the fastest
)

# Send 10 requests and watch which deployments get picked over time
print(f"{'Req':<6}{'Deployment':<32}{'Latency':<10}")
print("-" * 50)

for i in range(10):
    start = time.time()
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
        max_tokens=5
    )
    latency_ms = (time.time() - start) * 1000
    deployment = r._hidden_params.get("model_id", "?")
    print(f"#{i+1:<5}{deployment:<32}{latency_ms:>6.0f} ms")

Req   Deployment                      Latency   
--------------------------------------------------
#1    🔵 Gemini                          1298 ms
#2    🟢 Groq Llama-3.3                   267 ms
#3    🟢 Groq Llama-3.3                   283 ms
#4    🟢 Groq Llama-3.3                   270 ms
#5    🟢 Groq Llama-3.3                   170 ms
#6    🟢 Groq Llama-3.3                   179 ms
#7    🟢 Groq Llama-3.3                   190 ms
#8    🟢 Groq Llama-3.3                   159 ms
#9    🟢 Groq Llama-3.3                   185 ms
#10   🟢 Groq Llama-3.3                   176 ms


#### 🎯 Strategy 4: cost-based-routing — The "Always Cheapest" Pattern

The idea: Pick the deployment that costs the least per token right now. Beautiful for cost-sensitive apps.

In [23]:
import os
from litellm import Router

# Different providers with very different price points
model_list = [
    {"model_name": "chat",
     "litellm_params": {"model": "gemini/gemini-2.5-flash",             # ~$2.50/M input tokens
                        "api_key": os.getenv("GEMINI_API_KEY")},
     "model_info": {"id": "🔵 Gemini"}},
    {"model_name": "chat",
     "litellm_params": {"model": "ollama/gemma3:4b",        # ~$0.15/M input tokens
                        },
     "model_info": {"id": "🔵 ollama (cheap)"}},
    {"model_name": "chat",
     "litellm_params": {"model": "groq/llama-3.3-70b-versatile",   # ~$0.05/M
                        "api_key": os.getenv("GROQ_API_KEY")},
     "model_info": {"id": "🟢 Groq Llama (cheapest)"}},
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"   
)

for i in range(5):
    r = router.completion(
        model="chat",
        messages=[{"role": "user", "content": "Hi"}],
        max_tokens=10
    )
    print(f"Request {i+1} → {r._hidden_params.get('model_id', '?')}")

Request 1 → 🔵 Gemini
Request 2 → 🟢 Groq Llama (cheapest)
Request 3 → 🟢 Groq Llama (cheapest)
Request 4 → 🔵 ollama (cheap)
Request 5 → 🔵 ollama (cheap)


#### 📊 Part 9: Observability — Log Every Single Call

In production, you must log every LLM call: prompt, response, latency, cost, user_id, etc.

LiteLLM supports custom callbacks — here's a simple logger:

In [24]:
import litellm
from litellm import completion

# A simple in-memory log store
call_logs = []

def log_success(kwargs, completion_response, start_time, end_time):
    """Called automatically after every successful LLM call."""
    call_logs.append({
        "model": kwargs.get("model"),
        "prompt": kwargs["messages"][-1]["content"][:60],
        "input_tokens": completion_response.usage.prompt_tokens,
        "output_tokens": completion_response.usage.completion_tokens,
        "latency_sec": round((end_time - start_time).total_seconds(), 2),
        "cost_usd": kwargs.get("response_cost", 0),
        "user": kwargs.get("user", "anonymous")
    })

def log_failure(kwargs, completion_response, start_time, end_time):
    print("❌ Call failed:", kwargs.get("exception"))

# Register the callbacks
litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

# Make a few tagged calls
for q, user in [
    ("What is RAG?", "krish"),
    ("Explain transformers.", "student_42"),
    ("What is fine-tuning?", "krish"),
]:
    completion(
        model="groq/llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": q}],
        user=user  # tag the call for attribution
    )

# Review the audit log
import json
print(json.dumps(call_logs, indent=2, default=str))

[
  {
    "model": "llama-3.3-70b-versatile",
    "prompt": "What is RAG?",
    "input_tokens": 40,
    "output_tokens": 286,
    "latency_sec": 1.3,
    "cost_usd": 0.00024953999999999997,
    "user": "krish"
  },
  {
    "model": "llama-3.3-70b-versatile",
    "prompt": "Explain transformers.",
    "input_tokens": 39,
    "output_tokens": 888,
    "latency_sec": 2.56,
    "cost_usd": 0.0007245299999999999,
    "user": "student_42"
  }
]


#### 🔗 Part 10: Integrating the Gateway with LangChain

Here's where it really clicks for production GenAI apps:

LangChain for the orchestration (agents, chains, RAG) + LiteLLM as the unified LLM backend.

LangChain has a built-in ChatLiteLLM wrapper — drop it in like any other chat model.

In [26]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talks through LiteLLM
llm = ChatLiteLLM(model="groq/llama-3.3-70b-versatile", temperature=0.3)

# A standard LangChain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named GPT. Be concise."),
    ("user", "{question}")
])

# Compose with LCEL — same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

An LLM (Large Language Model) Gateway is:
* A platform that connects users to LLMs, enabling interaction and access to their capabilities.
* A layer of abstraction that simplifies the complexity of LLMs, making them more user-friendly and accessible.
* An interface that can manage multiple LLMs, route requests, and provide features like authentication, logging, and analytics.


#### 🤖 Part 11: A Real Example — Multi-Provider LangChain Chain with Fallbacks

Let's combine everything: a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks — and logs every call.

In [27]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Primary model
primary = ChatLiteLLM(model="gpt-x")

# Fallbacks (any LangChain-compatible model)
fallback_1 = ChatLiteLLM(model="ollama/gemma3:4b", temperature=0.2)
fallback_2 = ChatLiteLLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({"question": "What are the top 3 benefits of an LLM Gateway?"})
print(result)

❌ Call failed: litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=gpt-x
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers
```json
{
  "answer": [
    {
      "benefit_id": "1",
      "title": "Enhanced Security & Control",
      "description": "LLM Gateways provide a crucial layer of security by controlling access to your LLMs. This includes features like rate limiting, authentication/authorization (e.g., API keys, OAuth), input validation, and output filtering.  Without a gateway, you expose your models directly to potentially malicious or unintended usage, significantly reducing risk."
    },
    {
      "benefit_id": "2",
      "title": "Traffic Management & Scalability",
      "description": "Gateways handle incoming requests efficiently, preventing overload on your LLMs and ensuring consistent performan

#### 🧪 Part 13: A Mini End-to-End Demo — Smart Router for a Chatbot
Let's build a tiny **task-aware chatbot** that:

1. Decides what kind of question the user is asking (code, summary, general)
2. Routes to the right model accordingly
3. Falls back if the chosen model fails
4. Logs cost and latency

In [29]:
import time
from litellm import completion, completion_cost

def classify_task(user_query: str) -> str:
    """Cheap classifier — uses the fastest model to decide routing."""
    cls = completion(
        model="groq/llama-3.3-70b-versatile",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()


def call_with_fallbacks(model_chain, messages):
    """Try each model in order; return the first one that succeeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error

def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Each entry is a FULL chain: [primary, fallback1, fallback2, ...]
    # Every model name includes its provider prefix (groq/, anthropic/, etc.)
    routing = {
        "code":    [ "ollama/gemma3:4b",   "groq/llama-3.3-70b-versatile"],
        "summary": ["ollama/gemma3:4b",                "groq/llama-3.3-70b-versatile"],
        "general": ["groq/llama-3.3-70b-versatile", "ollama/gemma3:4b"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }

# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f"🏷️  Task:    {result['detected_task']}")
    print(f"🤖 Model:    {result['model_used']}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"💰 Cost:    {result['cost_usd']}")
    print(f"💬 Answer:  {result['answer'][:200]}...")

❓ Q: Write a Python function to compute Fibonacci numbers.
🏷️  Task:    code
🤖 Model:    ollama/gemma3:4b
⏱️  Latency: 0.0s
💰 Cost:    $0.000000
💬 Answer:  ```python
def fibonacci(n):
  """
  Computes the nth Fibonacci number using an iterative approach.

  Args:
    n: The index of the desired Fibonacci number (non-negative integer).
       The first Fi...
❓ Q: Summarize the importance of attention mechanism in 2 sentences.
🏷️  Task:    summary
🤖 Model:    ollama/gemma3:4b
⏱️  Latency: 0.0s
💰 Cost:    $0.000000
💬 Answer:  Attention mechanisms allow neural networks to focus on the most relevant parts of an input sequence, rather than treating all elements equally. This selective focusing dramatically improves performanc...
❓ Q: Tell me a fun fact about elephants.
🏷️  Task:    general
🤖 Model:    llama-3.3-70b-versatile
⏱️  Latency: 0.0s
💰 Cost:    n/a
💬 Answer:  Here's a fun fact about elephants: Elephants have a highly developed sense of empathy and can recognize themselves in a mirro